#### dissert — Kaggle worker template

Generic single-config launch template for XDash's Kaggle runner (EXPERIMENT_AUTOMATION_PLAN.md
§2.4/§2.5). **Do not edit or push this notebook directly** — the dashboard renders a copy of it
per launch, substituting the `# DASHBOARD:LAUNCH_SPEC` cell below, then pushes the rendered copy.
One push = one config, running train then eval sequentially inside this one kernel (a Kaggle
account only runs one kernel at a time, so there is no separate train-only/eval-only push to
chain a second half onto).

Environment setup (apt python3.8, a venv, pinned torch/mmcv wheels) and the private-repo clone
via a Kaggle secret are carried over from this repo's existing `notebooks/kaggle_run.ipynb`,
which is proven to work on this repo — this template does not reinvent that part, only adds the
dashboard integration (LAUNCH_SPEC marker, dataset auto-attach, two-stage run, result packaging)
around it.

**Not yet verified against a real Kaggle push** (unlike segpriors' template, which was). Verify
end-to-end before relying on it for an unattended batch.

## 1. GPU check

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import subprocess, time
from datetime import datetime, timezone
NOTEBOOK_START = time.time()  # XDASH_V2_PLAN.md §4.A7 -- measures actual setup time, not a guess
print(subprocess.run(["python3", "--version"], capture_output=True, text=True).stdout)


## 2. Clone the repo


In [ ]:
import os

username = "Syfur007"
repo_name = "dissert"

%cd /kaggle/working
if os.path.exists(repo_name):
    os.system(f"rm -rf {repo_name}")

!git clone https://github.com/{username}/{repo_name}.git
%cd {repo_name}


## 3. Reproduce the training environment

In [ ]:
!add-apt-repository ppa:deadsnakes/ppa -y
!apt-get update -qq
!apt-get install -y python3.8 python3.8-venv python3.8-dev
!python3.8 -m venv /kaggle/working/py38_env
PY = "/kaggle/working/py38_env/bin/python"
PIP = "/kaggle/working/py38_env/bin/pip"


In [ ]:
# --only-binary/--prefer-binary: falling back to a source build here can silently burn
# 20-40+ minutes of this notebook's own time budget (see EXPERIMENT_AUTOMATION_PLAN.md §8.1 —
# this setup time comes out of the same session as training and must be reserved for).
!{PIP} install --upgrade pip
!{PIP} install --only-binary=:all: SimpleITK==2.2.1
!{PIP} install --prefer-binary torch==1.13.1+cu117 torchvision==0.14.1+cu117 torchaudio==0.13.1 --extra-index-url https://download.pytorch.org/whl/cu117
!{PIP} install --prefer-binary mmcv-full -f https://download.openmmlab.com/mmcv/dist/cu117/torch1.13.0/index.html
!{PIP} install --prefer-binary -r requirements.txt


## 4. Launch spec — filled in by the dashboard

Left as-is (unsubstituted), the next cell fails loudly instead of silently running nothing.

In [ ]:
# DASHBOARD:LAUNCH_SPEC — substituted by backend/kaggle.py's _render_launch_notebook() before
# push. Do not hand-edit these values in this template file — edit a *pushed copy* only if
# you're debugging a specific run, never this shared template.
CONFIG_PATH = "__DASHBOARD_CONFIG_PATH__"
EXTRA_ARGS = "__DASHBOARD_EXTRA_ARGS__"
# Resolved server-side from this deployment's repo profile (repos/dissert.yaml's train_script/
# eval_script) — dissert's own train.py takes --seed/--seeds directly (see repos/dissert.yaml's
# seed_arg), so no sweep wrapper is needed the way segpriors needs one.
TRAIN_SCRIPT = "__DASHBOARD_TRAIN_SCRIPT__"
EVAL_SCRIPT = "__DASHBOARD_EVAL_SCRIPT__"
# Flags always appended to the eval command only — repos/dissert.yaml's eval_default_args is
# currently [] (see that file's own note: --ensemble there means fold-ensemble, not confirmed
# as a sane default yet), so this is normally empty for this repo.
EVAL_EXTRA_FLAGS = "__DASHBOARD_EVAL_EXTRA_FLAGS__"
# Kaggle dataset slug ("{username}/{slug}") resolved server-side by backend/dataset_map.py's
# resolve_kaggle_dataset() — the same value declared in kernel-metadata.json's dataset_sources
# for the declarative attach. May be empty if nothing resolved for this config's dataset name.
# Used below only as a fallback when the declarative attach didn't actually mount anything
# under /kaggle/input/.
DATASET_SOURCE = "__DASHBOARD_DATASET_SOURCE__"

assert CONFIG_PATH and not CONFIG_PATH.startswith("__DASHBOARD_"), (
    "CONFIG_PATH was never substituted -- this notebook was uploaded/run directly instead of "
    "pushed through the dashboard's Kaggle launch flow, which is what fills in this cell."
)

## 5. Attach the dataset this config needs

Generic by design: resolves `dataset.name`/`dataset.root` from the *composed* config the same
way `utils.config.load_config()` does for every other entry point, then looks for a same-named
directory already attached under `/kaggle/input/` and symlinks it into place. `load_config()`
returns a plain dict here (dissert's own schema — see `utils/config.py`, `orchestration/
schema.py`), unlike segpriors' pydantic-object config, so this indexes with `["dataset"]["root"]`
rather than attribute access.

If nothing's attached, falls back to pulling `DATASET_SOURCE` (the `{username}/{slug}` XDash
resolved server-side — see cell 4 above) straight from Kaggle via `kagglehub.dataset_download()`,
which is preinstalled on Kaggle's own kernel image and authenticates using the kernel's own
identity, no separate credentials needed. Only hard-fails if that also finds nothing: check
`configs/dataset/*.yaml` for what root layout the dataset is expected to have, and if
`DATASET_SOURCE` came back empty, map the dataset name in XDash's Data tab (or attach the right
Kaggle dataset via **+ Add Data** by hand) first.

In [ ]:
import subprocess, glob, shutil, json as _json

os.environ["PYTHONPATH"] = os.getcwd()
probe = subprocess.run(
    [PY, "-c",
     "from utils.config import load_config; import json; c = load_config(" + repr(CONFIG_PATH) + "); "
     "print(json.dumps({'name': c['dataset']['name'], 'root': c['dataset']['root']}))"],
    capture_output=True, text=True,
)
assert probe.returncode == 0, f"Could not resolve dataset info for {CONFIG_PATH}:\n{probe.stderr}"
ds = _json.loads(probe.stdout.strip().splitlines()[-1])
dataset_name, dataset_root = ds["name"], ds["root"]
print("dataset:", dataset_name, "-> root:", dataset_root)

def _find_leaf(base, leaf):
    return [p for p in glob.glob(f"{base}/**/{leaf}", recursive=True) if os.path.isdir(p)]

if not os.path.isdir(dataset_root):
    leaf = os.path.basename(dataset_root.rstrip("/"))
    candidates = _find_leaf("/kaggle/input", leaf)

    if not candidates and DATASET_SOURCE:
        # Declarative attach (kernel-metadata.json's dataset_sources) didn't mount anything --
        # pull it directly instead of failing, using the same slug XDash resolved server-side
        # (backend/dataset_map.py's resolve_kaggle_dataset()). kagglehub ships preinstalled on
        # Kaggle's kernel image and authenticates via the kernel's own identity.
        print(f"'{leaf}' not under /kaggle/input/ -- downloading '{DATASET_SOURCE}' via kagglehub")
        import kagglehub
        downloaded = kagglehub.dataset_download(DATASET_SOURCE)
        candidates = _find_leaf(downloaded, leaf) or ([downloaded] if os.path.isdir(downloaded) else [])

    assert candidates, (
        f"No '{leaf}' directory found under /kaggle/input/ for dataset '{dataset_name}' — "
        + (
            f"downloaded '{DATASET_SOURCE}' via kagglehub but it didn't contain a '{leaf}' "
            "directory either — check configs/dataset/*.yaml for the root layout it expects."
            if DATASET_SOURCE else
            "attach the matching Kaggle dataset via + Add Data, or map this dataset name in "
            "XDash's Data tab so the dashboard can attach/download it automatically."
        )
    )
    os.makedirs(os.path.dirname(dataset_root) or ".", exist_ok=True)
    os.symlink(candidates[0], dataset_root)
    print("symlinked", candidates[0], "->", dataset_root)

In [ ]:
# Fresh output dir for every push — a push must never inherit output from a previous kernel
# execution under the same kernel slug.
from pathlib import Path

run_root = Path("/kaggle/working/dissert")
outputs_root = run_root / "outputs"
if outputs_root.exists():
    shutil.rmtree(outputs_root)
outputs_root.mkdir(parents=True, exist_ok=True)


## 6. Run

Train, then eval — both inside this one kernel execution. Eval only runs if train exits 0,
mirroring the local scheduler's own skip-on-failure semantics for a `mode="both"` launch
(EXPERIMENT_AUTOMATION_PLAN.md §2.4).

Packaging (§7 below) always runs, in a `finally` block, whether train/eval succeed or not —
the previous version asserted on a nonzero exit code, which aborted the notebook before the
packaging cell ever ran, so a failed attempt shipped no artifacts and no diagnosis at all
(XDASH_V2_PLAN.md D10). `outputs/xdash_status.json` is what `backend/kaggle.py`'s `download()`
reads to tell a real failure from a successful run, instead of inferring it from zip presence.


In [ ]:
import shlex

started_at = datetime.now(timezone.utc).isoformat(timespec="seconds")
stage = "train"
returncode = None
try:
    cmd_train = [PY, TRAIN_SCRIPT, "--config", CONFIG_PATH] + (shlex.split(EXTRA_ARGS) if EXTRA_ARGS else [])
    print("Running (train):", " ".join(shlex.quote(c) for c in cmd_train))
    result = subprocess.run(cmd_train)
    returncode = result.returncode

    if returncode == 0:
        stage = "eval"
        eval_extra_flags = shlex.split(EVAL_EXTRA_FLAGS) if EVAL_EXTRA_FLAGS else []
        cmd_eval = [PY, EVAL_SCRIPT, "--config", CONFIG_PATH] + eval_extra_flags + (shlex.split(EXTRA_ARGS) if EXTRA_ARGS else [])
        print("Running (eval):", " ".join(shlex.quote(c) for c in cmd_eval))
        result = subprocess.run(cmd_eval)
        returncode = result.returncode
        if returncode == 0:
            stage = "done"
finally:
    # Unconditional — runs whether train/eval succeeded, failed, or something above raised.
    # A failed attempt with no packaged output and no recorded reason is undiagnosable from
    # the dashboard (XDASH_V2_PLAN.md D10) -- the previous version asserted on a nonzero exit
    # code, which aborted the notebook before the old, separate packaging cell ever ran.
    # Written *before* zipping so the status file itself lands inside outputs/ in the archive.
    xdash_status = {
        "stage": stage,
        "returncode": returncode,
        "started_at": started_at,
        "ended_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "setup_seconds": round(time.time() - NOTEBOOK_START, 1),
    }
    (outputs_root / "xdash_status.json").write_text(_json.dumps(xdash_status, indent=2))
    print("xdash status:", xdash_status)

    zip_result = subprocess.run(
        "cd /kaggle/working/dissert && zip -qr /kaggle/working/results.zip outputs/ -x '*.pth' "
        "&& find outputs -name '*.pth' | zip -q /kaggle/working/checkpoints.zip -@",
        shell=True,
    )
    print("packaging exit code:", zip_result.returncode)


## 7. Package results for download

Runs unconditionally, from the `finally` block above — not a separate cell that a failed run
could skip. dissert's own layout (OUTPUT_LAYOUT.md, `manifest_layout: "experiments"`):
everything lives under `outputs/` (nested per-experiment dirs plus the flat `outputs/ledger/`),
unlike segpriors' `artifacts/`+`logs/`+`checkpoints/` split — `backend/kaggle.py`'s
`register_ledger()` and `usage_history()` both branch on `manifest_layout` to find it at this
path inside the downloaded zip.


In [ ]:
# The zip step already ran inside cell 14's `finally` block above (so it happens even on a
# training/eval failure) -- this just confirms what landed.
!ls -lh /kaggle/working/*.zip
